# IVGD trên NetScience — chạy trên Colab GPU

Phần còn lại của bảng 5 baseline × 3 đồ thị (Review 30/10 Mục 7.4, nộp 10/11). IVGD trên CPU máy cá nhân ước tính 4–5 ngày cho NetScience (1.589 node), nên chạy ở đây.

**Trước khi chạy:** Runtime → Change runtime type → **GPU** (T4 là đủ). Chạy lần lượt từng ô. Giữ nguyên mọi tham số để cùng giao thức với máy cá nhân.

In [ ]:
# Ô 1: kiểm tra GPU
import torch, platform
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU - dừng lại, đổi runtime sang GPU")

In [ ]:
# Ô 2: cài đúng phiên bản như máy cá nhân (GraphSL 0.15, ndlib 6.0.1, torch-geometric 2.8.1).
# torch giữ bản có sẵn của Colab (bản CUDA); phiên bản ghi lại ở ô 6.
!pip -q install GraphSL==0.15 ndlib==6.0.1 torch-geometric==2.8.1 six

In [ ]:
# Ô 3: hàm chạy (chép logic run_baselines.py, chỉ IVGD)
import os, csv, time, random, datetime as dt, numpy as np, torch
from GraphSL.utils import load_dataset, diffusion_generation, split_dataset, download_dataset
from GraphSL.GNN.IVGD.main import IVGD
DIFF = dict(infect_prob=0.3, diff_type="IC", sim_num=100, seed_ratio=0.2)   # như máy cá nhân
os.makedirs("/content/graphsl", exist_ok=True); os.chdir("/content/graphsl")
if not os.path.exists("data"): download_dataset(os.getcwd())
LOG = "ivgd_colab.log"
def log(m):
    line = f"[{dt.datetime.now():%H:%M:%S}] {m}"; print(line, flush=True); open(LOG, "a").write(line + "\n")
def run_ivgd(ds, seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    adj, train, test = split_dataset(diffusion_generation(graph=load_dataset(ds, data_dir=os.getcwd()), **DIFF))
    log(f"== {ds} (n={adj.shape[0]}): {len(train)} train / {len(test)} test")
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    t0 = time.time(); m = IVGD(); diff = m.train_diffusion(adj, train)
    model, thres, *_ = m.train(adj, train, diff); t1 = time.time()
    r = m.test(adj, test, diff, model, thres); t2 = time.time()
    row = dict(time=dt.datetime.now().isoformat(timespec="seconds"), dataset=ds, n_nodes=adj.shape[0], method="IVGD",
               seed=seed, acc=r.acc, pr=r.pr, re=r.re, f1=r.f1, auc=r.auc, train_s=round(t1-t0,1), test_s=round(t2-t1,1),
               status="ok (colab " + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu") + ")")
    new = not os.path.exists("baselines_colab.csv")
    with open("baselines_colab.csv", "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row)); new and w.writeheader(); w.writerow(row)
    log(f"   IVGD {ds}: acc={r.acc:.3f} pr={r.pr:.3f} re={r.re:.3f} f1={r.f1:.3f} auc={r.auc:.3f} | train {t1-t0:.0f}s, test {t2-t1:.0f}s")
    return row

**Ô 4 — kiểm tra môi trường bằng Karate (vài phút).** Trên máy cá nhân IVGD-Karate cho acc 0,953 · pr 0,867 · re 0,867 · f1 0,867 · auc 0,986. Nếu Colab ra gần như vậy (GPU có thể lệch nhẹ ở chữ số thứ 3) thì môi trường khớp; nếu lệch nhiều, dừng lại và ghi chú trước khi chạy NetScience.

In [ ]:
run_ivgd("karate")

**Ô 5 — IVGD trên NetScience.** Theo dõi `Epoch [x/50]` (mô hình lan truyền) rồi `Epoch [x/200]` (IVGD). Nếu Colab ngắt kết nối giữa chừng thì chạy lại từ ô 3.

In [ ]:
run_ivgd("netscience")

In [ ]:
# Ô 6: ghi phiên bản và tải kết quả về
!pip freeze > requirements_colab.txt
from google.colab import files
for fn in ["baselines_colab.csv", "ivgd_colab.log", "requirements_colab.txt"]: files.download(fn)

Sau khi tải về: chép 3 file vào `C:\\Users\\ad\\Documents\\GraphSL_baselines\\results\\colab\\`. Trong bảng nộp, ô IVGD–NetScience ghi chú: "chạy trên Colab GPU (tên GPU), cùng giao thức; IVGD–Karate trên Colab dùng để đối chiếu môi trường".